In [ ]:
from getpass import getpass
import sys, os

REPO_NAME = "RecSys-Challenge-2025"
REPO_URL  = f"github.com/Lv1g1/{REPO_NAME}.git"
LOCAL_REPO_PATH = f"/content/{REPO_NAME}"

# Mount Google Drive first (Colab only)
if '/content' in os.getcwd():
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

# Clone only if not existing
if not os.path.exists(LOCAL_REPO_PATH):
    print("Cloning repo...")
    token = getpass("GitHub Token: ")
    !git clone https://{token}@{REPO_URL}
else:
    print("Repo already exists — pulling latest changes")
    os.chdir(LOCAL_REPO_PATH)
    !git pull
    os.chdir("/content")

# Enable import of your modules
if LOCAL_REPO_PATH not in sys.path:
    os.chdir(LOCAL_REPO_PATH)
    sys.path.append(os.getcwd())
    os.chdir("/content")

In [ ]:
import importlib
import scipy.sparse as sps
import optuna

from Challenge import paths
importlib.reload(paths)

from Evaluation.Evaluator import EvaluatorHoldout
from Challenge.hyper_tuning import hyperparameter_tuning

In [ ]:
# Load datasets
URM_train = sps.load_npz(paths.URM_TRAIN)
URM_validation = sps.load_npz(paths.URM_VALIDATION)

In [ ]:
# Set up evaluator
evaluator = EvaluatorHoldout(URM_validation, cutoff_list=[10])

In [ ]:
# Define objective function for hyperparameter tuning
from Recommenders.NonPersonalizedRecommender import GlobalEffects

def objective_function(optuna_trial: optuna.trial.Trial) -> float:
    recommender_instance = GlobalEffects(URM_train)
    recommender_instance.fit(
        # shrink factor
        lambda_user=optuna_trial.suggest_int("lambda_user", 0, 1000),
        lambda_item=optuna_trial.suggest_int("lambda_item", 0, 1000)
    )
    
    result_df, _ = evaluator.evaluateRecommender(recommender_instance)
    
    return result_df.loc[10]["MAP"]

In [ ]:
save_results, optuna_study = hyperparameter_tuning(objective_function, n_trials=50)

In [ ]:
# Train final model on train + validation with best hyperparameters
recommender = GlobalEffects(URM_train + URM_validation)
recommender.fit(
    lambda_user=optuna_study.best_trial.params["lambda_user"],
    lambda_item=optuna_study.best_trial.params["lambda_item"]
)